# Tourism Analytics - Model Training & Evaluation
## Complete ML Pipeline: Regression, Classification & Recommendation

This notebook covers:
- Loading preprocessed data
- Training regression models for rating prediction
- Training classification models for visit mode prediction
- Building recommendation systems
- Model evaluation and comparison

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')

from models import RatingPredictor, VisitModeClassifier, AttractionRecommender

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 1. Load Preprocessed Data

In [ ]:
# Load processed data
print("Loading processed data...")
df = pd.read_csv('../data/processed_data.csv')

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

## 2. Define Features for Modeling

In [ ]:
# Features for regression (predicting rating)
regression_features = [
    'UserId', 'AttractionId', 'VisitYear', 'VisitMonth', 'VisitQuarter',
    'ContinentId', 'RegionId', 'CountryId', 'CityId', 'AttractionCityId', 
    'AttractionTypeId', 'VisitMode',
    'UserAvgRating', 'UserRatingStd', 'UserTotalVisits', 'UserUniqueAttractions',
    'AttractionAvgRating', 'AttractionRatingStd', 'AttractionTotalVisits', 
    'AttractionUniqueVisitors', 'IsSummerMonth', 'IsWinterMonth', 'ModeTypeCount'
]

# Features for classification (predicting visit mode)
classification_features = [col for col in regression_features if 'VisitMode' not in col]

# Filter available features
regression_features = [f for f in regression_features if f in df.columns]
classification_features = [f for f in classification_features if f in df.columns]

print(f"Regression features: {len(regression_features)}")
print(f"Classification features: {len(classification_features)}")

## 3. Rating Prediction (Regression)
### Objective: Predict the rating a user will give to an attraction

In [ ]:
print("="*80)
print("RATING PREDICTION MODEL (REGRESSION)")
print("="*80)

# Initialize predictor
rating_predictor = RatingPredictor()

# Prepare data
X_train, X_test, y_train, y_test = rating_predictor.prepare_data(
    df, regression_features, target='Rating', test_size=0.2
)

In [ ]:
# Train multiple models
results = rating_predictor.train_models(X_train, y_train)

In [ ]:
# Evaluate best model
eval_results = rating_predictor.evaluate(X_test, y_test)

In [ ]:
# Feature importance
feature_importance = rating_predictor.get_feature_importance(top_n=15)

if feature_importance is not None:
    plt.figure(figsize=(10, 8))
    plt.barh(feature_importance['feature'], feature_importance['importance'])
    plt.xlabel('Importance')
    plt.title('Top 15 Most Important Features for Rating Prediction')
    plt.tight_layout()
    plt.show()
    
    print("\nTop Features:")
    print(feature_importance)

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(eval_results['actuals'], eval_results['predictions'], alpha=0.5)
plt.plot([1, 5], [1, 5], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title('Predicted vs Actual Ratings')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Save model
rating_predictor.save_model('../models/rating_predictor.pkl')

## 4. Visit Mode Classification
### Objective: Predict how a user will travel (Business, Family, Couples, Friends, Solo)

In [ ]:
print("\n" + "="*80)
print("VISIT MODE CLASSIFICATION MODEL")
print("="*80)

# Initialize classifier
visit_classifier = VisitModeClassifier()

# Prepare data
X_train_c, X_test_c, y_train_c, y_test_c = visit_classifier.prepare_data(
    df, classification_features, target='VisitMode', test_size=0.2
)

In [ ]:
# Train multiple models
results_c = visit_classifier.train_models(X_train_c, y_train_c)

In [ ]:
# Evaluate best model
eval_results_c = visit_classifier.evaluate(X_test_c, y_test_c)

In [ ]:
# Feature importance
feature_importance_c = visit_classifier.get_feature_importance(top_n=15)

if feature_importance_c is not None:
    plt.figure(figsize=(10, 8))
    plt.barh(feature_importance_c['feature'], feature_importance_c['importance'])
    plt.xlabel('Importance')
    plt.title('Top 15 Most Important Features for Visit Mode Classification')
    plt.tight_layout()
    plt.show()
    
    print("\nTop Features:")
    print(feature_importance_c)

In [ ]:
# Visualize confusion matrix
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(eval_results_c['confusion_matrix'], annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix for Visit Mode Classification')
plt.tight_layout()
plt.show()

In [ ]:
# Save model
visit_classifier.save_model('../models/visitmode_classifier.pkl')

## 5. Recommendation System
### Objective: Suggest attractions based on user preferences and similar users

In [ ]:
print("\n" + "="*80)
print("ATTRACTION RECOMMENDATION SYSTEM")
print("="*80)

# Initialize recommender
recommender = AttractionRecommender()

# Prepare data
user_item_matrix = recommender.prepare_data(df)

In [ ]:
# Build collaborative filtering model
recommender.build_collaborative_filtering()

In [ ]:
# Test collaborative filtering recommendations
sample_user = df['UserId'].iloc[100]
print(f"Testing collaborative filtering for User {sample_user}...")

collab_recs = recommender.recommend_collaborative(sample_user, top_n=5)

if collab_recs is not None and len(collab_recs) > 0:
    print("\nCollaborative Filtering Recommendations:")
    print(collab_recs)

In [ ]:
# Test content-based recommendations
print(f"\nTesting content-based filtering for User {sample_user}...")

content_recs = recommender.recommend_content_based(sample_user, top_n=5)

if content_recs is not None and len(content_recs) > 0:
    print("\nContent-Based Recommendations:")
    print(content_recs)

In [ ]:
# Test hybrid recommendations
print(f"\nTesting hybrid recommendations for User {sample_user}...")

hybrid_recs = recommender.recommend_hybrid(sample_user, top_n=5)

if hybrid_recs is not None and len(hybrid_recs) > 0:
    print("\nHybrid Recommendations:")
    print(hybrid_recs)

In [ ]:
# Save recommender
recommender.save_model('../models/recommender.pkl')

## 6. Model Performance Summary

In [ ]:
# Create performance summary
summary = {
    'Regression': {
        'Model': eval_results['model_name'],
        'R2 Score': f"{eval_results['r2']:.4f}",
        'RMSE': f"{eval_results['rmse']:.4f}",
        'MAE': f"{eval_results['mae']:.4f}"
    },
    'Classification': {
        'Model': eval_results_c['model_name'],
        'Accuracy': f"{eval_results_c['accuracy']:.4f}",
        'Precision': f"{eval_results_c['precision']:.4f}",
        'Recall': f"{eval_results_c['recall']:.4f}",
        'F1-Score': f"{eval_results_c['f1']:.4f}"
    }
}

print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY")
print("="*80)

print("\n1. RATING PREDICTION (REGRESSION):")
for key, value in summary['Regression'].items():
    print(f"   {key}: {value}")

print("\n2. VISIT MODE CLASSIFICATION:")
for key, value in summary['Classification'].items():
    print(f"   {key}: {value}")

print("\n3. RECOMMENDATION SYSTEM:")
print("   ✓ Collaborative Filtering implemented")
print("   ✓ Content-Based Filtering implemented")
print("   ✓ Hybrid Approach implemented")

In [ ]:
# Save summary to JSON
import json

with open('../models/model_performance.json', 'w') as f:
    json.dump(summary, f, indent=4)

print("\nModel performance summary saved to models/model_performance.json")

## 7. Conclusion

In [ ]:
print("\n" + "="*80)
print("MODEL TRAINING COMPLETED SUCCESSFULLY!")
print("="*80)

print("\nAll models have been trained, evaluated, and saved.")
print("\nNext Steps:")
print("1. Run the Streamlit application: streamlit run app.py")
print("2. Test predictions and recommendations")
print("3. Deploy the application")

print("\n✓ Ready for production!")